# Control difuso de climatización

Ejemplo complementario de la Unidad 2. Construye un sistema Mamdani y revisa su superficie de control.

## Dependencias

En un entorno nuevo, descomente y ejecute: `%pip install numpy matplotlib scikit-fuzzy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from skfuzzy import control as ctrl

np.set_printoptions(precision=3, suppress=True)

## Variables y membresías

El error se define como $e=T_{real}-T_{objetivo}$. Una acción positiva representa mayor enfriamiento.

In [ ]:
error = ctrl.Antecedent(np.linspace(-5, 5, 101), 'error')
delta = ctrl.Antecedent(np.linspace(-2, 2, 81), 'delta')
potencia = ctrl.Consequent(np.linspace(0, 100, 101), 'potencia')

error['negativo'] = fuzz.trimf(error.universe, [-5, -5, 0])
error['cero'] = fuzz.trimf(error.universe, [-2, 0, 2])
error['positivo'] = fuzz.trimf(error.universe, [0, 5, 5])

delta['bajando'] = fuzz.trimf(delta.universe, [-2, -2, 0])
delta['estable'] = fuzz.trimf(delta.universe, [-0.8, 0, 0.8])
delta['subiendo'] = fuzz.trimf(delta.universe, [0, 2, 2])

potencia['baja'] = fuzz.trimf(potencia.universe, [0, 0, 45])
potencia['media'] = fuzz.trimf(potencia.universe, [25, 50, 75])
potencia['alta'] = fuzz.trimf(potencia.universe, [55, 100, 100])

## Base de reglas

Las reglas buscan enfriar cuando el error es positivo y reducir potencia cuando la temperatura ya baja.

In [ ]:
reglas = [
    ctrl.Rule(error['negativo'], potencia['baja']),
    ctrl.Rule(error['cero'] & delta['bajando'], potencia['baja']),
    ctrl.Rule(error['cero'] & delta['estable'], potencia['media']),
    ctrl.Rule(error['cero'] & delta['subiendo'], potencia['alta']),
    ctrl.Rule(error['positivo'] & delta['bajando'], potencia['media']),
    ctrl.Rule(error['positivo'] & (delta['estable'] | delta['subiendo']), potencia['alta']),
]
sistema = ctrl.ControlSystem(reglas)

In [ ]:
def evaluar(error_value, delta_value):
    simulacion = ctrl.ControlSystemSimulation(sistema)
    simulacion.input['error'] = error_value
    simulacion.input['delta'] = delta_value
    simulacion.compute()
    return simulacion.output['potencia']

casos = [(-3.0, -0.5), (0.0, 0.0), (2.5, -1.0), (2.5, 1.0)]
for caso in casos:
    print(f'e={caso[0]:4.1f}, Δe={caso[1]:4.1f} -> potencia={evaluar(*caso):5.1f}%')

## Superficie de control

El barrido permite detectar discontinuidades, regiones sin cobertura y respuestas contrarias al conocimiento del dominio.

In [ ]:
x = np.linspace(-5, 5, 31)
y = np.linspace(-2, 2, 25)
xx, yy = np.meshgrid(x, y)
zz = np.array([[evaluar(e, d) for e in x] for d in y])

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(xx, yy, zz, cmap='viridis', edgecolor='none')
ax.set(xlabel='Error [°C]', ylabel='Cambio del error [°C/paso]', zlabel='Potencia [%]')
plt.show()

## Trabajo propuesto

1. Identifique una región donde cambiaría las reglas o membresías.
2. Agregue humedad como antecedente.
3. Defina pruebas automáticas para puntos nominales, límites y monotonicidad.